Attempts to extract taxonomy and impact direction at the same time as policy, but doesn't work.
The model seems to lost between the two tasks and does neither correctly.

In [11]:
import asyncio
import os
import time
from typing import Literal, Optional
from dotenv import load_dotenv

from openai import AsyncOpenAI
import pandas as pd
from pydantic import BaseModel, Field
from tqdm.notebook import tqdm

from pprint import pprint

load_dotenv()

True

In [2]:
client = AsyncOpenAI(
    base_url=os.getenv("GENERATION_API_URL"),
    #base_url="https://f3948611-98f8-4738-b973-30b4a657cffa.ifr.fr-par.scaleway.com/v1",
    api_key=os.getenv("SCW_SECRET_KEY"),
)

In [80]:
#df = pd.read_parquet('data/conclusions_sample_10k_2026-01-27.parquet')
df = pd.read_parquet('results_557k/sample_2000_extracted_policies.parquet')

In [81]:
df.policies.iloc[150]

array(['[MOBILITY] [REGULATORY] [NATIONAL] Policy framework for EV battery waste treatment in China, which includes targets for material recovery from waste EV batteries but lacks provisions for historical and orphan batteries and a target for battery collection.',
       '[MOBILITY] [REGULATORY] [NATIONAL] Policy framework for EV battery waste treatment in China, which aims to promote innovations in the technology, process, and practice of EV battery waste treatment and encourage their successful industrialization.',
       '[MOBILITY] [REGULATORY] [NATIONAL] Policy framework for EV battery waste treatment in China, which introduces regulatory standards to govern the process of waste battery treatment and aims to ensure the treatment of waste EV batteries through qualified channels.'],
      dtype=object)

In [8]:
df = df.drop(columns=["contains_policies", "policies"])
df

,openalex_id,chunk_idx,text
0,W4306385914,0,explore a new and effective method for mariti...
1,W3173622727,6,. Compositional analysis of lignocellulosic fe...
2,W4238379039,1,"2012), shown in overview, however the\n384 pre..."
3,W4408679953,0,affecting breastfeeding practice among mother...
4,W3092401747,1,for all ﬁve local sites in both August and No...
...,...,...,...
1995,W4379144753,0,study aimed to develop a combustion technolog...
1996,W4389428043,2,the context can be changed only to a limited ...
1997,W4313209439,0,a distinct signature of contemporary global w...
1998,W2408034890,0,Valplast® and DuraFlex™ are thermoplastic mat...


In [10]:
from dspy_policies_and_taxonomy_extraction.taxonomy_definition.impact_taxonomy import Resource, Wellbeing, Justice, PlanetaryBoundary

In [53]:
def enum_to_literal(enum_cls):
    return Literal[*list(enum_cls.__members__.keys())]

In [82]:
class JusticeImpact(BaseModel):
    dimension: enum_to_literal(Justice) = Field(..., description="The dimension of the impact on justice.")
    direction: Literal["positive", "negative", "neutral", "unknown"] = Field(..., description="The direction of the impact")


class WellbeingImpact(BaseModel):
    dimension: enum_to_literal(Wellbeing) = Field(..., description="The dimension of the impact on wellbeing.")
    direction: Literal["positive", "negative", "neutral", "unknown"] = Field(..., description="The direction of the impact")


class ResourceImpact(BaseModel):
    dimension: enum_to_literal(Resource) = Field(..., description="The dimension of the impact on resource.")
    direction: Literal["positive", "negative", "neutral", "unknown"] = Field(..., description="The direction of the impact")


class PlanetaryBoundaryImpact(BaseModel):
    dimension: enum_to_literal(PlanetaryBoundary) = Field(..., description="The dimension of the impact on planetary boundary.")
    direction: Literal["positive", "negative", "neutral", "unknown"] = Field(..., description="The direction of the impact")


class Impacts(BaseModel):
    planetary_boundary_impacts: list[PlanetaryBoundaryImpact] = Field(description="The impacts of the policy on planetary boundaries. List all dimensions mentioned in the text and only those.")
    resource_impacts: list[ResourceImpact] = Field(description="The impacts of the policy on resources, if mentioned in the text. List all dimensions mentioned in the text and only those.")
    wellbeing_impacts: list[WellbeingImpact] = Field(description="The impacts of the policy on wellbeing, if mentioned in the text. List all dimensions mentioned in the text and only those.")
    justice_impacts: list[JusticeImpact] = Field(description="The impacts of the policy on justice, if mentioned in the text. List all dimensions mentioned in the text and only those.")


class PolicyExtractionResponse(BaseModel):
    contains_policies: bool = Field(..., description="Whether the text mentions at least one policy.")
    policies: Optional[list[str]] = Field(..., description="If contains_policies is true, list of policies mentioned in the text.")
    impacts: Optional[Impacts] = Field(..., description="The impacts of each policy mentioned in policies, in the same order (we want a 1-1 mapping).")


model_name = "mistral-small-3.2-24b-instruct-2506"
#model_name = "qwen/qwen3-235b-a22b-instruct-2507"

with open('POLICIES_EXTRACTION_PROMPT.txt', 'r') as f:
    prompt = f.read()

In [83]:
PolicyExtractionResponse.model_json_schema()

{'$defs': {'Impacts': {'properties': {'planetary_boundary_impacts': {'description': 'The impacts of the policy on planetary boundaries. List all dimensions mentioned in the text and only those.',
     'items': {'$ref': '#/$defs/PlanetaryBoundaryImpact'},
     'title': 'Planetary Boundary Impacts',
     'type': 'array'},
    'resource_impacts': {'description': 'The impacts of the policy on resources, if mentioned in the text. List all dimensions mentioned in the text and only those.',
     'items': {'$ref': '#/$defs/ResourceImpact'},
     'title': 'Resource Impacts',
     'type': 'array'},
    'wellbeing_impacts': {'description': 'The impacts of the policy on wellbeing, if mentioned in the text. List all dimensions mentioned in the text and only those.',
     'items': {'$ref': '#/$defs/WellbeingImpact'},
     'title': 'Wellbeing Impacts',
     'type': 'array'},
    'justice_impacts': {'description': 'The impacts of the policy on justice, if mentioned in the text. List all dimensions men

In [84]:
async def extract_policies(
    text: str, prompt: str, model_name: str, client: AsyncOpenAI = client
) -> PolicyExtractionResponse:
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt.strip()},
                {"role": "user", "content": text},
            ],
            temperature=0,
            max_tokens=2048,
            response_format= {
                    "type": "json_schema",
                    "json_schema": {
                        "name": PolicyExtractionResponse.__name__,
                        "schema": PolicyExtractionResponse.model_json_schema(),
                    },
                },
            timeout=60,
            stream=False,
        )
        return PolicyExtractionResponse.model_validate_json(response.choices[0].message.content)
    except Exception as e:
        print('Error')
        return f"Error: {e}"


async def batch_extract_policies(
    texts: list[str], prompt: str, model_name: str, client: AsyncOpenAI = client
) -> list[PolicyExtractionResponse]:
    results = await asyncio.gather(*[extract_policies(text, prompt, model_name, client) for text in texts])
    return results

In [85]:
pprint(df.text.iloc[150])

(' on the policy review conducted in the previous sec- tion, this section '
 'discusses some of the major shortcom- ings of the existing policy framework '
 'for EV battery waste treatment in China.\n'
 'Lack of provisions for historical and orphan batter- ies: It has not been '
 'specified whether historical and or- phan batteries are included in the '
 'scope of the current policy framework. Historical batteries refer to waste '
 'bat- teries from EVs that have been sold prior to the imple- mentation of '
 'the current policy framework, and orphan batteries refer to waste batteries '
 'from EVs whose manu- facturers have ceased to exist. Though no reliable '
 'publicly available data on historical and orphan batteries exists in the '
 'Chinese EV market, it can be anticipated that this is- sue will become more '
 'pronounced in the next couple of years. As the government has started to cut '
 'EV subsidies since 2017, many EV makers (especially some loss-mak- ing EV '
 'start-ups) ha

In [79]:
res = await extract_policies(df.text.iloc[150], prompt, model_name, client)
res.policies

[PolicyWithImpacts(policy='[MATERIALS] [REGULATORY] [NATIONAL] Policy framework for EV battery waste treatment in China that does not include historical and orphan batteries in its scope.', planetary_boundary_impacts=[PlanetaryBoundaryImpact(dimension='climate_change', direction='negative')], resource_impacts=[ResourceImpact(dimension='non_metallic_minerals', direction='negative')], wellbeing_impacts=[WellbeingImpact(dimension='health', direction='negative')], justice_impacts=[JusticeImpact(dimension='procedural', direction='negative')]),
 PolicyWithImpacts(policy='[MATERIALS] [REGULATORY] [NATIONAL] Policy framework for EV battery waste treatment in China that does not set targets for the overall collection of waste batteries.', planetary_boundary_impacts=[PlanetaryBoundaryImpact(dimension='climate_change', direction='negative')], resource_impacts=[ResourceImpact(dimension='non_metallic_minerals', direction='negative')], wellbeing_impacts=[WellbeingImpact(dimension='health', direction

In [86]:
res = await extract_policies(df.text.iloc[150], prompt, model_name, client)
res.policies

['[MOBILITY] [REGULATORY] [NATIONAL] The current policy framework for EV battery waste treatment in China does not specify whether historical and orphan batteries are included in its scope.',
 '[MOBILITY] [REGULATORY] [NATIONAL] The current policy framework for EV battery waste treatment in China sets targets for material recovery from waste EV batteries but does not set a target for the overall collection of waste batteries.',
 '[MOBILITY] [REGULATORY] [NATIONAL] The current policy framework for EV battery waste treatment in China has introduced regulatory standards to govern the process of waste battery treatment, but the effective administration and enforcement of these standards is a shared responsibility between several central (e.g., MIIT, and SAMR) and local agencies without a clear definition of the scope of authority.']